<a href="https://colab.research.google.com/github/grlee1128/CMPCS-DS-410/blob/main/source.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_recall_curve, confusion_matrix
import numpy as np
import warnings
import os

# warnings
warnings.filterwarnings('ignore')

# 1. EFFICIENT DATA LOADING
print("--- Step 1: Loading Data Efficiently ---")

# Define columns we need to avoid Kernel Crash
cols_to_load = [
    'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK',
    'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST',
    'DISTANCE', 'CRS_DEP_TIME', 'DEP_DELAY',
    'CANCELLED', 'DIVERTED'
]

dfs = []
for i in range(1, 16):
    file_path = f"Filtered_part_{i}.csv"
    if os.path.exists(file_path):
        try:
            df_temp = pd.read_csv(file_path, usecols=cols_to_load)
            dfs.append(df_temp)
        except Exception as e:
            print(f"Error reading {file_path}: {e}")

if not dfs:
    print("CRITICAL ERROR: No files were loaded.")
else:
    df = pd.concat(dfs, ignore_index=True)
    print(f"Successfully combined files. Shape: {df.shape}")
    # 2. FEATURE ENGINEERING (OPTIMIZED)
    print("\n--- Step 2: Feature Engineering ---")

    # 2a. Filter & Create Target
    # Keep only valid flights
    df = df[(df['CANCELLED'] == 0) & (df['DIVERTED'] == 0)]
    df.dropna(subset=['DEP_DELAY'], inplace=True)

    # Target: 1 if delayed > 15 mins
    y = (df['DEP_DELAY'] > 15).astype(int)

    # 2b. Winter Holiday
    df['winter_holiday'] = 0
    mask_holiday = (df['MONTH'] == 12) | ((df['MONTH'] == 1) & (df['DAY_OF_MONTH'] <= 10))
    df.loc[mask_holiday, 'winter_holiday'] = 1

    # 2c. Time Features
    df['CRS_DEP_HOUR'] = df['CRS_DEP_TIME'] // 100

    bins = [-1, 5, 10, 15, 19, 24]
    labels = ['Late_Night', 'Morning', 'Midday', 'Afternoon_Evening', 'Night']
    df['TIME_OF_DAY'] = pd.cut(df['CRS_DEP_HOUR'], bins=bins, labels=labels, right=True)

    print("Features created.")

    # 3. LABEL ENCODING & PREPARATION
    print("\n--- Step 3: Encoding & Splitting ---")

    # Select Features
    features_to_use = [
        'MONTH', 'DAY_OF_WEEK', 'OP_UNIQUE_CARRIER',
        'ORIGIN', 'DEST', 'DISTANCE',
        'winter_holiday', 'CRS_DEP_HOUR', 'TIME_OF_DAY'
    ]

    X = df[features_to_use].copy()
    cat_cols = ['OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'TIME_OF_DAY']
    for col in cat_cols:
        X[col] = X[col].astype('category').cat.codes

    # Split Data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"Training Data Shape: {X_train.shape}")

    # 4. TRAINING XGBOOST
    print("\n--- Step 4: Training XGBoost ---")

    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        scale_pos_weight=2.5,
        n_estimators=200,
        max_depth=7,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    print("Model trained.")
    # 5. EVALUATION & OPTIMAL THRESHOLD
    print("\n--- Step 5: Finding Optimal Threshold ---")

    y_probs = model.predict_proba(X_test)[:, 1]

    # Calculate scores for all thresholds
    precision, recall, thresholds = precision_recall_curve(y_test, y_probs)

    # Calculate F1 only where valid (avoid divide by zero)
    numerator = 2 * recall * precision
    denominator = recall + precision
    f1_scores = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator!=0)

    best_idx = np.argmax(f1_scores)
    best_thresh = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]

    print(f"Optimal Threshold: {best_thresh:.4f}")
    print(f"Best F1 Score: {best_f1:.4f}")
    # 6. OUTPUT & CONFUSION MATRIX
    print("\n--- Step 6: Results ---")

    # Final Predictions based on optimal threshold
    y_pred_final = (y_probs > best_thresh).astype(int)

    # Classification Report
    print(classification_report(y_test, y_pred_final, target_names=['On-Time', 'Delayed']))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred_final)
    print("Confusion Matrix:")
    print(cm)

    tn, fp, fn, tp = cm.ravel()
    print(f"True Negatives (Correct On-Time): {tn}")
    print(f"False Positives (False Alarm): {fp}")
    print(f"False Negatives (Missed Delay): {fn}")
    print(f"True Positives (Caught Delay): {tp}")

    # Save Results to CSV
    results_df = pd.DataFrame({
        'Actual_Delayed': y_test,
        'Predicted_Delayed': y_pred_final,
        'Prediction_Probability': y_probs,
        'Correct_Prediction': (y_test == y_pred_final)
    })

    output_filename = 'confusion_matrix_predictions.csv'
    results_df.to_csv(output_filename, index=False)
    print(f"\nSaved detailed predictions to: {output_filename}")

    # 7. FEATURE IMPORTANCE
    print("\n--- Step 7: Feature Importance ---")
    importance_df = pd.DataFrame({
        'feature': X.columns,
        'importance': model.feature_importances_
    }).sort_values(by='importance', ascending=False)

    print(importance_df)
    print("\n--- Pipeline Complete ---")